# 29 - Context Optimization in RAG Pipelines with Langfuse Tracing
**Stack:** LangChain · FAISS · Langfuse · Ollama · Python 3.12
**Adapted from:** ServiceTitan interview prep session (May 2026)

This notebook builds intuition for the levers that matter in context optimization
and instruments everything with Langfuse so you can *see* the effect of each decision.

Levers covered:
- **Chunking strategy** -- fixed vs sentence vs semantic
- **Top-k retrieval** -- more context vs. noise floor
- **Reranking** -- cross-encoder quality improvement
- **Context window budget** -- token limits and compression

**Agent OS relevance:** This maps directly to the 'context and memory systems' pillar --
context freshness, retrieval quality, provenance, and observability.
Langfuse is self-hosted, making it suitable for air-gapped / compliance-sensitive deployments.

In [ ]:
# Install: pip install langchain langchain-community langchain-ollama
#           faiss-cpu sentence-transformers langfuse tiktoken rich

import os, time, json, uuid
from dataclasses import dataclass, field
from typing import Optional

OLLAMA_MODEL   = 'gpt-oss:20b'
OLLAMA_BASE    = 'http://localhost:11434'
LANGFUSE_HOST  = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')  # self-hosted
LANGFUSE_PK    = os.getenv('LANGFUSE_PUBLIC_KEY', 'MOCK')
LANGFUSE_SK    = os.getenv('LANGFUSE_SECRET_KEY', 'MOCK')
MOCK_MODE      = LANGFUSE_PK == 'MOCK'

print(f'Langfuse: {LANGFUSE_HOST} | Mock: {MOCK_MODE}')

## 1. Why Context Optimization Matters

The naive RAG approach: retrieve top-5 chunks, stuff into prompt, call LLM.
Problems:
- **Too few chunks:** miss relevant context, hallucination risk
- **Too many chunks:** hit token limit, dilute signal with noise
- **Wrong chunks:** wrong chunking strategy for the content type
- **Stale chunks:** indexed yesterday, system changed today

Context optimization is the systematic process of tuning these levers
using evaluation data -- not guessing.

In [ ]:
# Langfuse tracer (degrades gracefully to mock)
class MockTrace:
    def __init__(self, name, **kw): self.name = name
    def span(self, name, **kw): return MockSpan(name)
    def update(self, **kw): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass

class MockSpan:
    def __init__(self, name): self.name = name
    def update(self, **kw): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass

if not MOCK_MODE:
    try:
        from langfuse import Langfuse
        lf = Langfuse(public_key=LANGFUSE_PK, secret_key=LANGFUSE_SK, host=LANGFUSE_HOST)
        def new_trace(name, **kw): return lf.trace(name=name, **kw)
        print('Langfuse connected')
    except Exception as e:
        print(f'Langfuse unavailable ({e}) -- mock mode')
        def new_trace(name, **kw): return MockTrace(name)
else:
    def new_trace(name, **kw): return MockTrace(name)
    print('Langfuse mock mode (set env vars to connect to self-hosted instance)')

In [ ]:
# Build corpus and index (reusing patterns from notebook 28)
import re, math
from dataclasses import dataclass

CORPUS = [
    {'id': 'hvac', 'text': 'HVAC filters should be replaced every 90 days. Refrigerant levels should be checked annually. Condenser coils need cleaning each season. Electrical connections and lubrication are part of preventive maintenance.'},
    {'id': 'dispatch', 'text': 'Dispatch assigns technicians by proximity, skill, and workload. ML models predict job duration within 15% accuracy. Traffic integration reduces drive time by 25%. Real-time availability tracking improves first-call resolution.'},
    {'id': 'invoice', 'text': 'Digital invoicing cuts payment time to under 7 days. Itemized invoices with photos reduce disputes by 60%. Accounting software integration eliminates manual entry errors. Automatic reminders increase on-time payment by 30%.'},
    {'id': 'warranty', 'text': 'Warranty claims require proof of purchase and installation date. Labor warranties typically cover 90 days. Parts warranties vary by manufacturer from 1 to 10 years. Proper documentation prevents warranty disputes.'},
    {'id': 'compliance', 'text': 'EPA Section 608 requires certification for refrigerant handling. OSHA standards apply to electrical work and ladder safety. Local permits required for major HVAC replacements. Documentation of compliance reduces liability exposure.'},
]

# Chunking comparison
def sentence_chunks(text, n=2):
    sents = re.split(r'(?<=[.!?]) +', text)
    return [' '.join(sents[i:i+n]) for i in range(0, len(sents), n) if sents[i:i+n]]

def fixed_chunks(text, size=40, overlap=10):
    words = text.split()
    step = max(1, size - overlap)
    return [' '.join(words[i:i+size]) for i in range(0, len(words), step) if words[i:i+size]]

chunks_sent  = [(d['id'], c) for d in CORPUS for c in sentence_chunks(d['text'])]
chunks_fixed = [(d['id'], c) for d in CORPUS for c in fixed_chunks(d['text'])]

print(f'Sentence chunks: {len(chunks_sent)}')
print(f'Fixed chunks:    {len(chunks_fixed)}')
print(f'\nSample sentence chunk: {chunks_sent[0][1][:100]}')
print(f'Sample fixed chunk:    {chunks_fixed[0][1][:100]}')

In [ ]:
# Build FAISS index
try:
    from sentence_transformers import SentenceTransformer
    import faiss, numpy as np
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    def build_index(chunks_list):
        texts = [c[1] for c in chunks_list]
        embs  = embed_model.encode(texts, normalize_embeddings=True).astype('float32')
        idx   = faiss.IndexFlatIP(embs.shape[1])
        idx.add(embs)
        return idx, embs

    idx_sent,  _ = build_index(chunks_sent)
    idx_fixed, _ = build_index(chunks_fixed)
    EMBED_AVAILABLE = True
    print(f'Sentence index: {idx_sent.ntotal} vectors')
    print(f'Fixed index:    {idx_fixed.ntotal} vectors')
except ImportError as e:
    print(f'Embedding libs unavailable ({e}) -- mock retrieval')
    EMBED_AVAILABLE = False
    embed_model = idx_sent = idx_fixed = None

In [ ]:
# Retrieval with Langfuse tracing
def retrieve_with_trace(query: str, index, chunks_list, k: int, trace, span_name: str):
    t0 = time.time()
    with trace.span(span_name, input={'query': query, 'k': k}) as span:
        if EMBED_AVAILABLE:
            qvec = embed_model.encode([query], normalize_embeddings=True).astype('float32')
            scores, idxs = index.search(qvec, k)
            results = [{'chunk': chunks_list[i][1], 'doc_id': chunks_list[i][0], 'score': float(s)}
                       for i, s in zip(idxs[0], scores[0])]
        else:
            results = [{'chunk': chunks_list[i][1], 'doc_id': chunks_list[i][0], 'score': 0.8 - i*0.1}
                       for i in range(min(k, len(chunks_list)))]
        latency = (time.time() - t0) * 1000
        span.update(output={'retrieved': len(results), 'latency_ms': round(latency, 1)})
    return results

# Compare top-k values
query = 'What maintenance is required for HVAC systems?'
print(f'Query: {query!r}\n')
trace = new_trace('context_optimization_experiment', input={'query': query})

for k in [2, 4, 6]:
    results = retrieve_with_trace(query, idx_sent if EMBED_AVAILABLE else None, chunks_sent, k, trace, f'retrieve_k{k}')
    print(f'  k={k}: retrieved {len(results)} chunks')
    if results:
        print(f'  Top result (score={results[0]["score"]:.3f}): {results[0]["chunk"][:80]}...')
    print()

In [ ]:
# Context budget management
def count_tokens_approx(text: str) -> int:
    return len(text.split()) * 4 // 3  # rough approximation

def build_context_within_budget(
    query: str, chunks_list, index,
    max_tokens: int = 1500, top_k: int = 8
) -> dict:
    candidates = retrieve_with_trace(query, index, chunks_list, top_k, MockTrace('budget'), 'retrieve')
    selected   = []
    token_total = count_tokens_approx(query)

    for c in candidates:
        chunk_tokens = count_tokens_approx(c['chunk'])
        if token_total + chunk_tokens <= max_tokens:
            selected.append(c)
            token_total += chunk_tokens
        else:
            break

    context_text = '\n'.join(f'[{c["doc_id"]}]: {c["chunk"]}' for c in selected)
    return {
        'context': context_text,
        'chunks_used': len(selected),
        'chunks_available': len(candidates),
        'tokens_used': token_total,
        'token_budget': max_tokens,
    }

print('Context budget experiment:')
for budget in [500, 1000, 1500, 2000]:
    result = build_context_within_budget(
        query, chunks_sent, idx_sent if EMBED_AVAILABLE else None, max_tokens=budget
    )
    print(f'  Budget {budget:4d} tokens -> {result["chunks_used"]}/{result["chunks_available"]} chunks, {result["tokens_used"]} tokens used')

In [ ]:
# Freshness control pattern
import time

@dataclass
class ChunkWithMetadata:
    doc_id: str
    text: str
    indexed_at: float = field(default_factory=time.time)
    source_updated_at: Optional[float] = None
    ttl_seconds: int = 3600  # 1 hour default

    @property
    def is_fresh(self) -> bool:
        age = time.time() - self.indexed_at
        return age < self.ttl_seconds

    @property
    def needs_reindex(self) -> bool:
        if self.source_updated_at and self.source_updated_at > self.indexed_at:
            return True
        return not self.is_fresh

# Demo
stale_chunk = ChunkWithMetadata(doc_id='invoice', text='Old invoice text',
    indexed_at=time.time() - 7200, ttl_seconds=3600)
fresh_chunk = ChunkWithMetadata(doc_id='hvac', text='Fresh HVAC text')

print(f'Stale chunk: is_fresh={stale_chunk.is_fresh}, needs_reindex={stale_chunk.needs_reindex}')
print(f'Fresh chunk: is_fresh={fresh_chunk.is_fresh}, needs_reindex={fresh_chunk.needs_reindex}')

In [ ]:
print('=' * 60)
print('Notebook 29: Context Optimization with Langfuse Tracing')
print('=' * 60)
print('''
Demonstrated:
  [x] Chunking strategy comparison (sentence vs fixed)
  [x] Top-k retrieval experiments with Langfuse trace spans
  [x] Context window budget management (token-aware selection)
  [x] Freshness control via TTL and source_updated_at
  [x] Self-hosted Langfuse integration (mock-graceful fallback)

Agent OS JD mapping:
  Context assembly          -> build_context_within_budget()
  Freshness controls        -> ChunkWithMetadata.needs_reindex
  Provenance                -> doc_id tracking through pipeline
  Retrieval quality metrics -> score comparison across k values
  Observability (Langfuse)  -> per-step trace spans with latency

Self-hosted note:
  docker run -d -p 3000:3000 langfuse/langfuse
  Set LANGFUSE_HOST=http://localhost:3000 to enable tracing
''')